In [21]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/processed/train_processed.csv')

train_df.head()

,engine_id,cycle,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,125
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,125
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,125
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,125


In [22]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

scaler = MinMaxScaler()

train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

In [23]:
print(train_df.shape)
print(train_df.columns)

(20631, 19)
Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'sensor_2',
       'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21', 'RUL'],
      dtype='object')


Creating Sequences

In [24]:
def create_sequences(df, seq_length, feature_cols):
    sequences = []
    labels = []

    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]

        data = engine_data[feature_cols].values
        rul = engine_data['RUL'].values

        for i in range(len(data) - seq_length):
            sequences.append(data[i:i+seq_length])
            labels.append(rul[i+seq_length])

    return np.array(sequences), np.array(labels)

In [25]:
feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

In [26]:
sequence_length = 30

X, y = create_sequences(train_df, sequence_length, feature_cols)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17631, 30, 16)
y shape: (17631,)


In [27]:
print(X[0].shape)
print(y[0])

(30, 16)
125


Now doing the test/train split

note: 
I am doing engine wise split

In [28]:
from sklearn.model_selection import train_test_split

# get unique engine ids
engine_ids = train_df['engine_id'].unique()

# split engines (not rows)
train_ids, val_ids = train_test_split(engine_ids, test_size=0.2, random_state=42)

Filtering data

In [29]:
train_data = train_df[train_df['engine_id'].isin(train_ids)]
val_data = train_df[train_df['engine_id'].isin(val_ids)]

Creating sequences again

In [30]:
X_train, y_train = create_sequences(train_data, 30, feature_cols)
X_val, y_val = create_sequences(val_data, 30, feature_cols)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (14161, 30, 16)
Val: (3470, 30, 16)


Now loading test data and rul one and doing the same on them 

In [31]:
# original column names (same as initial train loading)
columns = ['engine_id', 'cycle'] + \
          [f'op_setting_{i}' for i in range(1, 4)] + \
          [f'sensor_{i}' for i in range(1, 22)]

test_df = pd.read_csv('../CMAPSSData/test_FD001.txt', sep=' ', header=None)
test_df = test_df.dropna(axis=1)

test_df.columns = columns

In [32]:
test_df = test_df.drop(columns=[
    'sensor_1', 'sensor_5', 'sensor_10',
    'sensor_16', 'sensor_18', 'sensor_19',
    'op_setting_3', 'sensor_6'
])

In [33]:
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

Creating sequences

In [34]:
def get_last_sequences(df, seq_length, feature_cols):
    sequences = []

    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]

        data = engine_data[feature_cols].values

        sequences.append(data[-seq_length:])

    return np.array(sequences)

X_test_final = get_last_sequences(test_df, 30, feature_cols)

print(X_test_final.shape)

(100, 30, 16)


In [35]:
X_test_final = get_last_sequences(test_df, 30, feature_cols)

print(X_test_final.shape)

(100, 30, 16)


In [36]:
rul_test = pd.read_csv('../CMAPSSData/RUL_FD001.txt', header=None)

print(rul_test.shape)

(100, 1)


In [37]:
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/y_train.npy', y_train)

np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/y_val.npy', y_val)

np.save('../data/processed/X_test.npy', X_test_final)
np.save('../data/processed/rul_test.npy', rul_test.values)